# Ensemble Methods

**Topic:** Supervised Learning — Model Combination

In [ ]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import ipywidgets as widgets
from ipywidgets import Dropdown, IntSlider, Output, HBox, VBox
from IPython.display import display, clear_output
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import (RandomForestClassifier,
                               GradientBoostingClassifier,
                               VotingClassifier, BaggingClassifier)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
np.random.seed(42)
from tkh_utils import PALETTE, FONT, base_layout


---
## What you'll explore

By the end of this demo you will be able to:

- **Describe** the three main ensemble strategies: bagging, boosting, and stacking
- **Explain** why combining multiple weak learners almost always outperforms any single one
- **Interpret** an ensemble convergence chart showing how accuracy improves as more models are added

> **Tip:** In the widget below, switch the strategy dropdown between Bagging, Boosting, and Voting and drag the estimator slider. Watch each curve climb quickly at first, then flatten out well above the dashed single-tree baseline — extra estimators keep costing more compute but stop buying much more accuracy.

---
## How we got here

Ensemble methods are the "meta" layer on top of the algorithms you have already studied:

- **[supervised/07_decision_trees.ipynb](07_decision_trees.ipynb)** — the base learner for both Random Forest (bagging) and Gradient Boosting; understanding how a single tree splits and overfits is prerequisite
- **[supervised/08_random_forests.ipynb](08_random_forests.ipynb)** — the canonical bagging algorithm; this notebook extends that idea to the full ensemble landscape
- **[supervised/09_gradient_boosting.ipynb](09_gradient_boosting.ipynb)** — the canonical boosting algorithm; the contrast with bagging is the central theme here

---
## Why this matters for data science

Ensemble methods almost always outperform single models. The reason is mathematical: averaging many independent predictors reduces variance without increasing bias. The practical lesson is that when you need maximum accuracy on a well-defined supervised task, an ensemble is almost always the right choice.

Understanding the three strategies — bagging, boosting, stacking — also gives you the vocabulary to read modern ML research and to combine your own specialized models in production systems.

---
## Where it sits on the spectrum

See **[ml_concepts/13_interpretability_vs_complexity.ipynb](../ml_concepts/13_interpretability_vs_complexity.ipynb)** for the full spectrum.

Ensembles do not occupy one fixed point — they inherit the spectrum from whichever base learner they wrap, then shift it further toward complexity and away from interpretability. A single decision tree is easy to read top to bottom; a Random Forest of 200 trees or a Gradient Boosting ensemble of 200 sequential stumps cannot be inspected tree by tree. Voting and stacking ensembles push even further right, since they combine predictions from structurally different model types.

What you gain back is feature importance and partial dependence, which restore a coarser, aggregate form of interpretability even when the individual trees are opaque.

---
## Try it yourself

In [ ]:
out = Output()
caption = widgets.HTML()

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=10,
                            n_informative=5, random_state=42)

strategy_dropdown = Dropdown(
    options=["Bagging", "Boosting", "Voting"], value="Bagging",
    description="Strategy:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="300px"),
)
n_slider = IntSlider(
    value=50, min=1, max=100, step=1,
    description="Estimators:",
    style={"description_width": "80px"},
    layout=widgets.Layout(width="420px"),
    continuous_update=False,
)

estimator_counts = list(range(1, 101, 5))

STRATEGY_NOTES = {
    "Bagging": "bootstrap-resampled trees plus random feature subsets (Random Forest)",
    "Boosting": "shallow trees trained sequentially, each correcting the previous one's errors",
    "Voting": "bootstrap-resampled trees combined purely by majority vote, no feature subsampling",
}

def build_model(strategy, n):
    if strategy == "Bagging":
        return RandomForestClassifier(n_estimators=n, random_state=42)
    elif strategy == "Boosting":
        return GradientBoostingClassifier(
            n_estimators=n, max_depth=1, learning_rate=0.1, random_state=42)
    else:
        return BaggingClassifier(
            estimator=DecisionTreeClassifier(), n_estimators=n,
            bootstrap=True, random_state=42)

single_tree_acc = cross_val_score(
    DecisionTreeClassifier(random_state=42), X, y, cv=5, scoring="accuracy"
).mean()

accuracy_curves = {
    strategy: [
        cross_val_score(build_model(strategy, n), X, y, cv=5, scoring="accuracy").mean()
        for n in estimator_counts
    ]
    for strategy in ["Bagging", "Boosting", "Voting"]
}

def render(change=None):
    strategy = strategy_dropdown.value
    n_value = n_slider.value
    accs = accuracy_curves[strategy]
    idx = int(np.argmin(np.abs(np.array(estimator_counts) - n_value)))

    fig = go.Figure()
    fig.add_trace(go.Scatter(
        x=estimator_counts, y=accs, mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2.5), marker=dict(size=6),
        name=f"{strategy} ensemble",
    ))
    fig.add_trace(go.Scatter(
        x=[estimator_counts[idx]], y=[accs[idx]], mode="markers",
        marker=dict(size=16, color=PALETTE["accent"], symbol="circle-open",
                    line=dict(width=3)),
        name=f"Slider position (N={estimator_counts[idx]})",
    ))
    fig.add_trace(go.Scatter(
        x=[estimator_counts[0], estimator_counts[-1]], y=[single_tree_acc] * 2,
        mode="lines", line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        name=f"Single tree: {single_tree_acc:.3f}",
    ))

    fig.update_layout(**{k: v for k, v in base_layout(
        title=(f"{strategy} — N≈{estimator_counts[idx]} estimators, "
               f"accuracy = {accs[idx]:.3f}"),
        xaxis_title="Number of Estimators",
        yaxis_title="5-Fold CV Accuracy",
    ).to_plotly_json().items()})
    fig.update_layout(height=440, yaxis=dict(range=[0.5, 1.0]))

    with out:
        clear_output(wait=True)
        display(go.FigureWidget(fig))

    gain = accs[idx] - single_tree_acc
    caption.value = (
        f"<b>{strategy}</b> ({STRATEGY_NOTES[strategy]}) reaches "
        f"<b>{accs[idx]:.1%}</b> accuracy at N≈{estimator_counts[idx]} estimators — "
        f"{gain:+.1%} versus the single-tree baseline of {single_tree_acc:.1%}. "
        f"Notice how little the curve moves past roughly N=20: most of the benefit "
        f"arrives early, and the rest of the slider mostly costs compute."
    )

strategy_dropdown.observe(render, names="value")
n_slider.observe(render, names="value")

display(VBox([HBox([strategy_dropdown, n_slider]), out, caption]))
render()

---
## What's happening?

**Bagging** (Bootstrap Aggregating): each model trains on a different random subset of the data. Predictions are combined by majority vote (classification) or averaging (regression). The models are built independently and can be trained in parallel. Random Forest is the most successful bagging algorithm.

**Boosting**: each model trains on the residuals (errors) of the previous one. Models are built sequentially, each correcting the last. The final prediction is a weighted sum. Gradient Boosting and XGBoost are the most successful boosting algorithms.

**Stacking**: base models (diverse: a tree, an SVM, logistic regression) each make predictions. A second-level meta-model learns how to best combine those predictions. Stacking is more complex but can extract value from diverse model types that bagging and boosting cannot.

| Strategy | Variance reduction | Bias reduction | Parallelizable | Example |
|---|---|---|---|---|
| Bagging | Yes | No | Yes | Random Forest |
| Boosting | Yes | Yes | No | Gradient Boosting, XGBoost |
| Stacking | Yes | Partial | Partial | Custom blends |

---
## When to use it / When NOT to use it

| Use it when | Do NOT use it when |
|---|---|
| You need the best possible accuracy on a well-defined supervised task | You need to explain individual predictions to a non-technical stakeholder |
| The base learner overfits (bagging) or underfits (boosting) on its own | Training time and compute budget are tightly constrained |
| You can afford the extra training and inference cost of many models | The model must be small enough to deploy on constrained hardware |
| You have a strong single model and want incremental accuracy gains via stacking | A single simple model already meets the accuracy bar — added complexity buys nothing |
| Kaggle-style competitions or offline batch scoring where latency does not matter | Real-time inference latency is critical and a single model is fast enough |

---
## Real-world example: Weak learners become strong together

A single shallow decision tree (depth=1) has high bias — it can only ask one question. Adding many of these "stumps" through boosting eventually builds a powerful classifier. The chart shows accuracy as stumps are added one at a time.

- **Notice:** The first few stumps improve accuracy dramatically; later ones add smaller and smaller improvements
- **Notice:** A single depth-1 stump barely beats random guessing; 200 of them combined are competitive with any single deep tree
- **Notice:** The boosting curve is monotonically decreasing on training loss — training error never goes up with more rounds (but test error can)

> **Discussion question:** Bagging and boosting both use many trees. What is the fundamental difference in how they use those trees? When would you choose bagging over boosting?

### Ensemble methods summary

| Method | How it combines | Bias reduction | Variance reduction | Examples |
|---|---|---|---|---|
| Bagging | Majority vote / average | No | Yes | Random Forest, Extra Trees |
| Boosting | Weighted additive | Yes | Yes | GBM, XGBoost, AdaBoost |
| Stacking | Meta-model on predictions | Partial | Yes | Custom blends, AutoML |
| Voting | Simple vote / weighted vote | No | Yes | VotingClassifier |

In [ ]:
from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import cross_val_score
import numpy as np, plotly.graph_objects as go

np.random.seed(42)
X, y = make_classification(n_samples=500, n_features=10,
                            n_informative=5, random_state=42)

estimator_counts = list(range(1, 101, 5))
boost_accs = []
for n in estimator_counts:
    m = GradientBoostingClassifier(
        n_estimators=n, learning_rate=0.1,
        max_depth=1, random_state=42
    )
    score = cross_val_score(m, X, y, cv=5, scoring='accuracy').mean()
    boost_accs.append(score)

single_stump = cross_val_score(
    DecisionTreeClassifier(max_depth=1, random_state=42),
    X, y, cv=5, scoring='accuracy'
).mean()

fig = go.Figure(data=[
    go.Scatter(
        x=estimator_counts, y=boost_accs, mode="lines+markers",
        line=dict(color=PALETTE["primary"], width=2.5),
        marker=dict(size=7), name="Boosted ensemble (depth-1 stumps)",
    ),
    go.Scatter(
        x=[estimator_counts[0], estimator_counts[-1]],
        y=[single_stump, single_stump], mode="lines",
        line=dict(color=PALETTE["secondary"], width=2, dash="dash"),
        name=f"Single stump: {single_stump:.3f}",
    ),
], layout=base_layout(
    title="Ensemble Strength: Accuracy vs Number of Boosted Stumps",
    xaxis_title="Number of Estimators",
    yaxis_title="5-Fold CV Accuracy",
))
fig.update_layout(yaxis=dict(range=[0.5, 1.0]))
fig.show()

> **Ensemble methods combine many weak learners into a strong one by reducing variance (bagging), reducing bias (boosting), or both — and the combined model almost always outperforms any individual model.**

---
*Next up: 14 — Multiclass Classification, extending binary classifiers to problems with three or more classes*